# **Setup**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/FYP")
ENG_DIR  = BASE_DIR / "data" / "engineered"
NLP_DIR  = BASE_DIR / "data" / "nlp"
GDELT_DIR  = BASE_DIR / "data" / "GDELT"
PANEL_DIR = BASE_DIR / "data" / "panel"

print("Engineered dir:", ENG_DIR)
print("NLP dir:", NLP_DIR)
print("GDELT dir:", GDELT_DIR)
print("Panel dir:", PANEL_DIR)

Engineered dir: /content/drive/MyDrive/FYP/data/engineered
NLP dir: /content/drive/MyDrive/FYP/data/nlp
GDELT dir: /content/drive/MyDrive/FYP/data/GDELT
Panel dir: /content/drive/MyDrive/FYP/data/panel


In [ ]:
# Destination countries (modelling focus)
DEST_ISO3 = ["USA", "MYS", "IDN"]

# Modelling windows
TARGET_YEARS = [2000, 2005, 2010, 2015, 2020, 2024]

DEST_ISO3, TARGET_YEARS

(['USA', 'MYS', 'IDN'], [2000, 2005, 2010, 2015, 2020, 2024])

# **Loads**

## Load ethnic composition (base)

In [ ]:
eth_path = ENG_DIR / "engineered_ethnic_composition_full.csv"
eth = pd.read_csv(eth_path)

print(eth.head())
print(eth.columns.tolist())

  iso3  year ethnic_group  value_interp  share_interp  total_pop_interp
0  ARG  2000      African        229472      0.005030        45618787.0
1  ARG  2000   Indigenous       1306730      0.028645        45618787.0
2  ARG  2000   Not Stated      44082585      0.966325        45618787.0
3  ARG  2005      African        229472      0.005030        45618787.0
4  ARG  2005   Indigenous       1306730      0.028645        45618787.0
['iso3', 'year', 'ethnic_group', 'value_interp', 'share_interp', 'total_pop_interp']


In [ ]:
eth["iso3"] = eth["iso3"].astype(str)
eth["year"] = eth["year"].astype(int)

eth = eth[
    eth["iso3"].isin(DEST_ISO3) &
    eth["year"].isin(TARGET_YEARS)
].copy()

# Keep core columns; adapt if you have 'total_pop' or 'eth_count'
keep_cols = ["iso3", "year", "ethnic_group", "share_interp"]
for extra in ["share", "total_pop", "eth_count", "count"]:
    if extra in eth.columns and extra not in keep_cols:
        keep_cols.append(extra)

eth = eth[keep_cols].copy()
eth.rename(columns={"iso3": "dest_iso3"}, inplace=True)
eth.sort_values(["dest_iso3", "ethnic_group", "year"], inplace=True)

print("Ethnic base panel shape:", eth.shape)
eth.head()

Ethnic base panel shape: (132, 4)


,dest_iso3,year,ethnic_group,share_interp
4836,IDN,2000,"Banjar, Melayu Banjar",0.017386
4845,IDN,2005,"Banjar, Melayu Banjar",0.028743
4854,IDN,2010,"Banjar, Melayu Banjar",0.040099
4863,IDN,2015,"Banjar, Melayu Banjar",0.040099
4872,IDN,2020,"Banjar, Melayu Banjar",0.040099


## Load IMS destination and origin features (in/out flows)

In [ ]:
ims_dest_path = ENG_DIR / "ims_destination_features.csv"
ims_orig_path = ENG_DIR / "ims_origin_features.csv"

ims_dest = pd.read_csv(ims_dest_path)
ims_orig = pd.read_csv(ims_orig_path)

print("IMS destination features cols:", ims_dest.columns.tolist())
print("IMS origin features cols     :", ims_orig.columns.tolist())

IMS destination features cols: ['iso3', 'year', 'dest_total_migrant_stock', 'n_origins', 'origin_hhi']
IMS origin features cols     : ['iso3', 'year', 'orig_total_migrant_stock', 'n_destinations', 'destination_hhi']


In [ ]:
# Destination features (inflow structure)
ims_dest["iso3"] = ims_dest["iso3"].astype(str)
ims_dest["year"] = ims_dest["year"].astype(int)

ims_dest = ims_dest[
    ims_dest["iso3"].isin(DEST_ISO3) &
    ims_dest["year"].isin(TARGET_YEARS)
].copy()

ims_dest.rename(columns={"iso3": "dest_iso3"}, inplace=True)

# Optional: prefix for clarity
dest_feature_cols = [c for c in ims_dest.columns if c not in ["dest_iso3", "year"]]
ims_dest.rename(columns={c: f"ims_dest_{c}" for c in dest_feature_cols}, inplace=True)

print("IMS DEST shape:", ims_dest.shape)
ims_dest.head()


IMS DEST shape: (18, 5)


,dest_iso3,year,ims_dest_dest_total_migrant_stock,ims_dest_n_origins,ims_dest_origin_hhi
0,IDN,2000,272821.0,27,0.472872
1,IDN,2005,90418.0,27,0.183197
2,IDN,2010,96493.0,27,0.182467
3,IDN,2015,133132.0,27,0.194085
4,IDN,2020,290640.0,27,0.222346


In [ ]:
# Origin features (outflow from the same country when it acts as origin)
ims_orig["iso3"] = ims_orig["iso3"].astype(str)
ims_orig["year"] = ims_orig["year"].astype(int)

ims_orig = ims_orig[
    ims_orig["iso3"].isin(DEST_ISO3) &
    ims_orig["year"].isin(TARGET_YEARS)
].copy()

ims_orig.rename(columns={"iso3": "dest_iso3"}, inplace=True)

orig_feature_cols = [c for c in ims_orig.columns if c not in ["dest_iso3", "year"]]
ims_orig.rename(columns={c: f"ims_orig_{c}" for c in orig_feature_cols}, inplace=True)

print("IMS ORIG (dest as origin) shape:", ims_orig.shape)
ims_orig.head()


IMS ORIG (dest as origin) shape: (18, 5)


,dest_iso3,year,ims_orig_orig_total_migrant_stock,ims_orig_n_destinations,ims_orig_destination_hhi
0,IDN,2000,2187871.0,63,0.197991
1,IDN,2005,2470619.0,63,0.242542
2,IDN,2010,3106885.0,63,0.269970
3,IDN,2015,3351461.0,63,0.254731
4,IDN,2020,3546661.0,63,0.252859


## Load GDELT

In [ ]:
gdelt_path = GDELT_DIR / "gdelt_country_window_wide_hybridwindows_volatility.csv"
gdelt = pd.read_csv(gdelt_path)

# Dtypes + cleaning
gdelt["iso3"] = gdelt["iso3"].astype(str).str.strip().str.upper()
gdelt["year"] = gdelt["year"].astype(int)

# Restrict to modelling windows
gdelt = gdelt[gdelt["year"].isin(TARGET_YEARS)].copy()

# Destination-side drivers
drivers_dest = gdelt[gdelt["iso3"].isin(DEST_ISO3)].copy()
drivers_dest = drivers_dest.rename(columns={"iso3": "dest_iso3"})
drivers_dest = drivers_dest.sort_values(["dest_iso3", "year"]).reset_index(drop=True)

# Origin-side drivers (for push aggregation)
# Origins can appear in flows even if not in DEST_ISO3, so keep all iso3 that appear in flows origins later.
drivers_orig = gdelt.copy().rename(columns={"iso3": "orig_iso3"})
drivers_orig = drivers_orig.sort_values(["orig_iso3", "year"]).reset_index(drop=True)

print("GDELT drivers DEST shape:", drivers_dest.shape)
print("GDELT drivers ORIG shape:", drivers_orig.shape)
print("DEST cols:", drivers_dest.columns.tolist())


GDELT drivers DEST shape: (18, 41)
GDELT drivers ORIG shape: (318, 41)
DEST cols: ['dest_iso3', 'year', 'gdelt_events_Aid/Cooperation', 'gdelt_events_Conflict/Migration', 'gdelt_events_Disapproval/Sanction', 'gdelt_events_Other', 'gdelt_events_Protest', 'gdelt_goldstein_Aid/Cooperation', 'gdelt_goldstein_Conflict/Migration', 'gdelt_goldstein_Disapproval/Sanction', 'gdelt_goldstein_Other', 'gdelt_goldstein_Protest', 'gdelt_tone_Aid/Cooperation', 'gdelt_tone_Conflict/Migration', 'gdelt_tone_Disapproval/Sanction', 'gdelt_tone_Other', 'gdelt_tone_Protest', 'conflict_events', 'conflict_tone', 'conflict_goldstein', 'coop_events', 'coop_tone', 'coop_goldstein', 'conflict_events_d1', 'conflict_events_d2', 'coop_events_d1', 'coop_events_d2', 'conflict_tone_d1', 'conflict_tone_d2', 'coop_tone_d1', 'coop_tone_d2', 'conflict_goldstein_d1', 'conflict_goldstein_d2', 'coop_goldstein_d1', 'coop_goldstein_d2', 'conflict_events_z', 'conflict_spike_hi', 'conflict_spike_med', 'push_multiplier', 'coop_even

# **Build DEST-YEAR table + Build CORE**

Build DEST-YEAR table (key spine)

In [ ]:
dest_years = eth[["dest_iso3", "year"]].drop_duplicates().copy()

Build CORE destination features (IMS only)

In [ ]:
dest_features_core = dest_years.merge(
    ims_dest,
    on=["dest_iso3", "year"],
    how="left"
).merge(
    ims_orig,
    on=["dest_iso3", "year"],
    how="left"
)

print("dest_features_core shape:", dest_features_core.shape)

dest_features_core shape: (18, 8)


Build CORE PANEL = Ethnic + IMS only

In [ ]:
panel_core = eth.merge(
    dest_features_core,
    on=["dest_iso3", "year"],
    how="left"
)

panel_core = panel_core.sort_values(["dest_iso3", "ethnic_group", "year"]).copy()

def add_lags(df, group_cols, target_cols, max_lag=2):
    df = df.copy()
    df = df.sort_values(group_cols + ["year"])
    for tcol in target_cols:
        for lag in range(1, max_lag + 1):
            df[f"{tcol}_lag{lag}"] = (
                df.groupby(group_cols)[tcol]
                  .shift(lag)
            )
    return df

# Lags only for ethnic share (and optional count columns if already present in eth)
lag_targets = ["share_interp"]
for extra in ["eth_count", "count"]:
    if extra in panel_core.columns and extra not in lag_targets:
        lag_targets.append(extra)

panel_core = add_lags(panel_core, ["dest_iso3", "ethnic_group"], lag_targets, max_lag=2)

# Key uniqueness enforcement
panel_core = (
    panel_core.sort_values(["dest_iso3", "ethnic_group", "year"])
              .drop_duplicates(subset=["dest_iso3", "ethnic_group", "year"], keep="last")
)

core_path = PANEL_DIR / "panel_core_ethnic_ims.csv"
panel_core.to_csv(core_path, index=False)
print("Saved CORE panel:", core_path, "| shape:", panel_core.shape)

# Sanity: no gdelt_ columns
gdelt_in_core = [c for c in panel_core.columns if "gdelt" in c.lower()]
print("GDELT cols in CORE:", len(gdelt_in_core))

Saved CORE panel: /content/drive/MyDrive/FYP/data/panel/panel_core_ethnic_ims.csv | shape: (132, 12)
GDELT cols in CORE: 0


# **Build GDELT**

In [ ]:
gdelt_dest = drivers_dest.copy()
gdelt_dest_cols = [c for c in gdelt_dest.columns if c not in ["dest_iso3", "year"]]
gdelt_dest = gdelt_dest.rename(columns={c: f"gdelt_dest_{c}" for c in gdelt_dest_cols})

Build GDELT origin drivers (ORIG) and origin-weighted push aggregation

In [ ]:
flows_path = ENG_DIR / "ims_od_inflow_outflow_engineered.csv"
flows = pd.read_csv(flows_path)

# Basic cleaning (match your earlier logic)
flows["iso3_dest"] = flows["iso3_dest"].astype(str).str.strip().str.upper()
flows["iso3_orig"] = flows["iso3_orig"].astype(str).str.strip().str.upper()
flows["year"] = flows["year"].astype(int)

flows = flows[
    flows["iso3_dest"].isin(DEST_ISO3) &
    flows["year"].isin(TARGET_YEARS)
].copy()

# Decide weight column (update if needed)
weight_col = "migrant_stock"
if weight_col not in flows.columns:
    raise ValueError(f"{weight_col} not found in flows columns: {flows.columns.tolist()}")

# Prefix ORIG gdelt features
gdelt_orig = drivers_orig.copy()
gdelt_orig_cols = [c for c in gdelt_orig.columns if c not in ["orig_iso3", "year"]]
gdelt_orig = gdelt_orig.rename(columns={c: f"gdelt_orig_{c}" for c in gdelt_orig_cols})

# Merge origin gdelt onto flows (per origin-year)
flows_g = flows.merge(
    gdelt_orig,
    left_on=["iso3_orig", "year"],
    right_on=["orig_iso3", "year"],
    how="left"
)

# Weighted aggregation: for each dest-year,
# compute weighted mean of origin gdelt signals
# weight = migrant_stock
w = flows_g[weight_col].astype(float).replace(0.0, np.nan)

push_cols = [c for c in flows_g.columns if c.startswith("gdelt_orig_")]

def weighted_mean(group, cols, wcol):
    ww = group[wcol].astype(float)
    out = {}
    denom = np.nansum(ww)
    for c in cols:
        x = group[c].astype(float)
        if denom == 0 or np.isnan(denom):
            out[c] = np.nan
        else:
            out[c] = np.nansum(x * ww) / denom
    return pd.Series(out)

dest_origpush = (
    flows_g.groupby(["iso3_dest", "year"], as_index=False)
           .apply(lambda g: weighted_mean(g, push_cols, weight_col))
)

dest_origpush = dest_origpush.rename(columns={"iso3_dest": "dest_iso3"})

print("dest_origpush shape:", dest_origpush.shape)

dest_origpush shape: (18, 41)


/tmp/ipython-input-4020571147.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: weighted_mean(g, push_cols, weight_col))


Build GDELT-ENRICHED destination features

In [ ]:
dest_features_gdelt = dest_years.merge(
    gdelt_dest,
    on=["dest_iso3", "year"],
    how="left"
).merge(
    dest_origpush,
    on=["dest_iso3", "year"],
    how="left"
)

print("dest_features_gdelt shape:", dest_features_gdelt.shape)

dest_features_gdelt shape: (18, 80)


Build PANEL_GDELT = CORE + GDELT drivers

In [ ]:
panel_gdelt = panel_core.merge(
    dest_features_gdelt,
    on=["dest_iso3", "year"],
    how="left"
)

panel_gdelt = (
    panel_gdelt.sort_values(["dest_iso3", "ethnic_group", "year"])
               .drop_duplicates(subset=["dest_iso3", "ethnic_group", "year"], keep="last")
)

gdelt_path = PANEL_DIR / "panel_core_ethnic_ims_gdelt.csv"
panel_gdelt.to_csv(gdelt_path, index=False)
print("Saved GDELT panel:", gdelt_path, "| shape:", panel_gdelt.shape)

Saved GDELT panel: /content/drive/MyDrive/FYP/data/panel/panel_core_ethnic_ims_gdelt.csv | shape: (132, 90)


# **Save conflict/coop column lists**

In [ ]:
gdelt_cols_all = [c for c in panel_gdelt.columns if c.startswith("gdelt_")]

conflict_cols = [c for c in gdelt_cols_all if "conflict" in c.lower() or "fight" in c.lower() or "war" in c.lower()]
coop_cols     = [c for c in gdelt_cols_all if "coop" in c.lower() or "cooper" in c.lower() or "peace" in c.lower()]

(pd.DataFrame({"col": sorted(conflict_cols)})
   .to_csv(PANEL_DIR / "conflict_cols.csv", index=False))
(pd.DataFrame({"col": sorted(coop_cols)})
   .to_csv(PANEL_DIR / "coop_cols.csv", index=False))

print("Saved:", PANEL_DIR / "conflict_cols.csv", "| n:", len(conflict_cols))
print("Saved:", PANEL_DIR / "coop_cols.csv", "| n:", len(coop_cols))

Saved: /content/drive/MyDrive/FYP/data/panel/conflict_cols.csv | n: 30
Saved: /content/drive/MyDrive/FYP/data/panel/coop_cols.csv | n: 26


Final sanity checks

In [ ]:
print("CORE has GDELT?", len([c for c in panel_core.columns if c.startswith("gdelt_")]) > 0)
print("GDELT panel has GDELT?", len([c for c in panel_gdelt.columns if c.startswith("gdelt_")]) > 0)
print("Any NLP cols in either?",
      len([c for c in panel_core.columns if c.startswith("nlp_")]),
      len([c for c in panel_gdelt.columns if c.startswith("nlp_")]))

CORE has GDELT? False
GDELT panel has GDELT? True
Any NLP cols in either? 0 0


# **Delta driver table for inflow modeling**

In [ ]:
conflict_cols = pd.read_csv(PANEL_DIR / "conflict_cols.csv")["col"].dropna().astype(str).tolist()
coop_cols     = pd.read_csv(PANEL_DIR / "coop_cols.csv")["col"].dropna().astype(str).tolist()

# Split into dest vs orig sets
conflict_dest = [c for c in conflict_cols if c.startswith("gdelt_dest_")]
conflict_orig = [c for c in conflict_cols if c.startswith("gdelt_orig_")]
coop_dest     = [c for c in coop_cols     if c.startswith("gdelt_dest_")]
coop_orig     = [c for c in coop_cols     if c.startswith("gdelt_orig_")]

# Build one row per dest_iso3-year
delta_tbl = (
    panel_gdelt[["dest_iso3", "year"] + conflict_dest + conflict_orig + coop_dest + coop_orig]
    .drop_duplicates(subset=["dest_iso3", "year"])
    .copy()
)

# Aggregate signals (using sum, can use mean as well)
delta_tbl["dest_conflict"] = delta_tbl[conflict_dest].sum(axis=1, skipna=True) if conflict_dest else 0.0
delta_tbl["dest_coop"]     = delta_tbl[coop_dest].sum(axis=1, skipna=True)     if coop_dest else 0.0
delta_tbl["orig_conflict"] = delta_tbl[conflict_orig].sum(axis=1, skipna=True) if conflict_orig else 0.0
delta_tbl["orig_coop"]     = delta_tbl[coop_orig].sum(axis=1, skipna=True)     if coop_orig else 0.0

# dest_coop + orig_conflict = positive pull into destination
# dest_conflict + orig_coop = negative effect on destination
delta_tbl["positive_signal"] = delta_tbl["dest_coop"] + delta_tbl["orig_conflict"]
delta_tbl["negative_signal"] = delta_tbl["dest_conflict"] + delta_tbl["orig_coop"]
delta_tbl["delta_driver"] = delta_tbl["positive_signal"] - delta_tbl["negative_signal"]

out_path = PANEL_DIR / "delta_driver_country_year.csv"
delta_tbl[["dest_iso3","year","dest_conflict","dest_coop","orig_conflict","orig_coop","positive_signal","negative_signal","delta_driver"]].to_csv(out_path, index=False)
print("Saved:", out_path, "| shape:", delta_tbl.shape)


Saved: /content/drive/MyDrive/FYP/data/panel/delta_driver_country_year.csv | shape: (18, 65)
